# 02 · Herramientas con MCP

**Bloque:** 0:20–0:45 · **Práctica:** 0:27–0:45

### Teoría breve
- **MCP (Model Context Protocol / Protocolo de Contexto de Modelo):** estándar para conectar modelos con herramientas externas.
- **Arquitectura:** *Cliente MCP* (tu agente) ⇄ *Servidor MCP* (expone tools) ⇄ *Herramientas* (APIs, archivos...).

Usaremos el servidor de ejemplo en `mcp_server/server.py`, que expone dos tools: `calcular` y `clima`.

In [ ]:
import sys, os
REPO = os.path.abspath("..")
if REPO not in sys.path:
    sys.path.insert(0, REPO)
from config import get_chat_model, get_embeddings
print("Entorno cargado ✔")

## Paso 1 · Ver el servidor MCP
Ábrelo en el editor: `mcp_server/server.py`. Aquí solo mostramos sus tools.

In [ ]:
print(open(os.path.join(REPO, "mcp_server", "server.py"), encoding="utf-8").read())

## Paso 2 · Conectar el cliente MCP
El cliente lanza el servidor como subproceso (transporte `stdio`) y descubre sus tools.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "taller": {
        "command": sys.executable,
        "args": [os.path.join(REPO, "mcp_server", "server.py")],
        "transport": "stdio",
    }
})

tools = await client.get_tools()
print("Herramientas MCP disponibles:", [t.name for t in tools])

## Paso 3 · Agente que usa las tools de MCP
El mismo `create_agent`, pero ahora sus herramientas vienen del servidor MCP.

In [ ]:
from langchain.agents import create_agent

agente = create_agent(get_chat_model(), tools=tools,
    system_prompt="Eres un asistente que usa herramientas para calcular y consultar el clima.")

r = await agente.ainvoke({"messages": [
    {"role": "user", "content": "¿Cuánto es 15 * (3 + 2)? Y dime qué clima hace en Madrid."}
]})
print(r["messages"][-1].content)

## ✅ Checkpoint 2
Muestra el **rastro de mensajes** para evidenciar que el agente llamó a las herramientas.

In [ ]:
for m in r["messages"]:
    m.pretty_print()